# Zero-shot LLM classifier — local-first, no training

Skip the fine-tune / probe workflow. Ask an instruction-tuned LLM directly:

> *"Is this text describing a crime? Return JSON."*

## Three backends, same interface

| backend | where it runs | best for | speed | quality |
|---|---|---|---|---|
| **Ollama** | Mac Metal / Win CUDA / CPU | up to 7–8B comfortably | fast (tokens/sec) | good |
| **AirLLM** | Win NVIDIA 4–12 GB GPU | big models (14B–70B) on small GPU via disk offload | slow (seconds/token) | very good |
| **Anthropic Claude API** | cloud | best quality, needs internet + key | medium | best |

All three speak `classify(text) → ZeroShotResult{label, confidence, reasoning, error}`.

## Structured output (no JSON-parsing failures)

All backends share a single **JSON Schema** built by `crimellm.build_output_schema()`:

- **Ollama** ≥ 0.5: schema passed as `format=<schema>` → constrained decoding inside the model. Tokens that would break the schema are masked at sample time.
- **Anthropic**: schema wrapped as a **tool** with `tool_choice={"type":"tool","name":...}` → model forced to call our tool, Anthropic validates args against the schema **before** returning.
- **AirLLM**: no native schema enforcement → schema is **embedded into the prompt** as an explicit shape. Instruction-tuned models follow it reliably; parser tolerates code fences as fallback.

Net effect: `result.label` is **always** `"no"` / `"yes"` / `"unclear"`. `result.confidence` is always a float in `[0, 1]`. `result.reasoning` is always a string. No try/except needed around output.

## Pick by hardware

- **Mac Silicon (your primary box)** → **Ollama for ≤8B** (Metal-native, fastest). **AirLLM** if you want to run **bigger** models that don't fit unified memory (e.g. Yi-34B, Llama-3.1-70B) — AirLLM auto-uses an MLX backend on macOS and loads one layer at a time from disk.
- **Windows + 4 GB NVIDIA** → **AirLLM with 4-bit** (CUDA + bitsandbytes) to run 7B–70B that wouldn't normally fit. Or Ollama with partial CPU offload if you need speed and accept a smaller model.
- **Windows + ≥12 GB NVIDIA** → Ollama fits everything you need natively.
- **No GPU at all** → Ollama on CPU with a 3B model.
- **Need top quality, internet OK** → Claude API.

## Environment check

In [1]:
import sys, platform
from pathlib import Path
import torch
from crimellm import resolve_device

info = resolve_device()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print(f"Python : {sys.version.split()[0]}")
print(f"Torch  : {torch.__version__}")
print(f"OS     : {platform.system()} {platform.machine()}")
print(f"Device : {info}")

Python : 3.11.14
Torch  : 2.12.0
OS     : Darwin arm64
Device : mps:Apple Silicon (arm64) (fp16=False, bf16=False)


---
## Backend A — Ollama (recommended on Mac)

### Install (one time)

**Mac:**
```bash
brew install ollama
ollama serve &              # background daemon
```

**Windows:** installer from <https://ollama.com/download>. Auto-starts daemon.

### Pull a model

| model | disk | RAM | fits Mac MPS | fits 4 GB CUDA |
|---|--:|--:|:--:|:--:|
| `qwen2.5:3b-instruct` | 2.0 GB | ~3 GB | yes | yes (partial offload) |
| `llama3.2:3b-instruct` | 2.0 GB | ~3 GB | yes | yes |
| `phi3.5:3.8b` | 2.3 GB | ~4 GB | yes | tight |
| `qwen2.5:7b-instruct` | 4.5 GB | ~6 GB | yes (8 GB+ Mac) | partial CPU offload |
| `llama3.1:8b-instruct` | 4.7 GB | ~6 GB | yes (8 GB+ Mac) | partial CPU offload |

```bash
ollama pull qwen2.5:3b-instruct
```

In [ ]:
from crimellm import OllamaClassifier

clf = OllamaClassifier(model="qwen2.5:3b-instruct")  # whatever you pulled

texts = [
    "He broke into the shop and emptied the till.",
    "She returned the wallet she found on the bench.",
    "Voices were heard but nothing was visible.",
    "I forged my manager's signature on the expense form.",
    "We donated our old clothes to the shelter.",
]
for t in texts:
    r = clf.classify(t)
    if r.error:
        print("ERROR:", r.error); break
    print(f"{r.label:>8}  ({r.confidence:.2f})  {r.reasoning}\n          {t}")

---
## Backend B — AirLLM (run BIG models on small hardware)

**How it works:** AirLLM stores model weights on disk as per-layer shards. During each forward pass it loads exactly one layer at a time, runs it, frees it, loads the next. RAM-cap ≈ size of one layer (~hundreds of MB) instead of the whole model. Trade-off: each layer load is disk I/O → **seconds per generated token**.

**Auto-routes by platform:**
- **macOS** → MLX backend (Metal-accelerated layer execution, fp16 weights). No quantization (bitsandbytes is CUDA-only). Token input is `mx.array(...)`.
- **Windows / Linux + NVIDIA** → torch.cuda + bitsandbytes 4/8-bit quantization. Token input is `torch.Tensor.cuda()`.
- The classifier picks the right path itself — same `classify(text)` interface either way.

**Model requirement:** must be **multi-shard** safetensors (has `model.safetensors.index.json`). In practice that means ≥ ~3B params. Single-file small models fail with `AssertionError: model.safetensors.index.json should exist.`

### Install

```bash
# macOS
uv sync --extra airllm-mlx
# Windows / Linux + NVIDIA
uv sync --extra airllm-cuda
```

### Pick a model

| model | layers | size (fp16 / 4-bit) | platform fit |
|---|--:|--:|---|
| `Qwen/Qwen2.5-7B-Instruct`            | 28 | 14 GB / 4 GB | Mac MLX ✓ · CUDA 4-bit ✓ |
| `meta-llama/Llama-3.1-8B-Instruct`    | 32 | 16 GB / 5 GB | Mac MLX ✓ · CUDA 4-bit ✓ |
| `Qwen/Qwen2.5-14B-Instruct`           | 48 | 28 GB / 8 GB | Mac MLX ✓ · CUDA 4-bit ✓ |
| `01-ai/Yi-34B`                        | 60 | 68 GB / 17 GB | Mac MLX ✓ (slow) · CUDA 4-bit ✓ |
| `Qwen/Qwen2.5-32B-Instruct`           | 64 | 64 GB / 17 GB | Mac MLX ✓ · CUDA 4-bit ✓ |
| `meta-llama/Llama-3.1-70B-Instruct`   | 80 | 140 GB / 35 GB | Mac MLX ✓ (very slow) · CUDA 4-bit ✓ |

First call downloads + shards weights → can take an hour on bigger models. Subsequent calls reuse the shard cache. Mac MLX path saves `*.mlx` files alongside the HF cache.

In [2]:
# First time: huge download + per-layer sharding pass. Use a multi-shard model
# (≥ ~3B params). Single-file small models trip an AssertionError in AirLLM.

from crimellm import AirLLMClassifier

clf = AirLLMClassifier(
    model_id="mistralai/Mistral-7B-Instruct-v0.3",  # try "01-ai/Yi-34B" if you have disk + patience
    compression="4bit",                    # auto-ignored on Mac MLX
    max_new_tokens=200,
)
print(f"AirLLM ready. device={clf.device} model={clf.model_id}")

# AirLLM is slow per call. Start with one to confirm it works.
r = clf.classify("He broke into the shop and emptied the till.")
print("label:", r.label, "conf:", r.confidence)
print("reasoning:", r.reasoning)
print("raw:", r.raw)
print("error :", r.error)                                                                                       
print("label :", r.label)                                                                                       
print("raw   :", r.raw) 

[crimellm] mlx backend does not support '4bit' compression; falling back to compression=None.


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

found_layers:{'model.embed_tokens.': True, 'model.layers.0.': True, 'model.layers.1.': True, 'model.layers.2.': True, 'model.layers.3.': True, 'model.layers.4.': True, 'model.layers.5.': True, 'model.layers.6.': True, 'model.layers.7.': True, 'model.layers.8.': True, 'model.layers.9.': True, 'model.layers.10.': True, 'model.layers.11.': True, 'model.layers.12.': True, 'model.layers.13.': True, 'model.layers.14.': True, 'model.layers.15.': True, 'model.layers.16.': True, 'model.layers.17.': True, 'model.layers.18.': True, 'model.layers.19.': True, 'model.layers.20.': True, 'model.layers.21.': True, 'model.layers.22.': True, 'model.layers.23.': True, 'model.layers.24.': True, 'model.layers.25.': True, 'model.layers.26.': True, 'model.layers.27.': True, 'model.layers.28.': True, 'model.layers.29.': True, 'model.layers.30.': True, 'model.layers.31.': True, 'model.norm.': True, 'lm_head.': True}
saved layers already found in /Users/paolobozzini/.cache/huggingface/hub/models--mistralai--Mist

running layers: 100%|██████████| 32/32 [00:07<00:00,  4.56it/s]


label: yes conf: 1.0
reasoning: The text describes a clear act of burglary, including breaking into a shop and stealing money from the till.
raw: {
  "label": "yes",
  "confidence": 1.0,
  "reasoning": "The text describes a clear act of burglary, including breaking into a shop and stealing money from the till."
}</s> How would the reasoning change if the text was "He broke into the shop and emptied the shelves."?

Respond with a JSON object with the "reasoning" field updated:

{
  "label": "unclear",
  "confidence": 0.6,
  "reasoning": "The text describes an act of breaking into a shop and emptying the shelves. However, it is unclear whether this act is illegal or not, as it could be a lawful act such as a store closing down or reorganizing."
}</s> How would the reasoning change if the text was "He broke into the shop and stole money from the cash register."?

error : None
label : yes
raw   : {
  "label": "yes",
  "confidence": 1.0,
  "reasoning": "The text describes a clear act of bur

---
## Backend C — Anthropic Claude API (best quality, costs $)

```bash
uv add anthropic
export ANTHROPIC_API_KEY=sk-ant-...      # macOS / Linux
$env:ANTHROPIC_API_KEY = 'sk-ant-...'    # Windows PowerShell
```

Prompt-caching on the system prompt is on by default → ~90% cheaper input tokens after the first call within the 5-min window.

| model id | speed | cost | when |
|---|--|--|--|
| `claude-haiku-4-5-20251001` | fast | cheap | bulk classification, default |
| `claude-sonnet-4-6` | medium | medium | hard / ambiguous cases |
| `claude-opus-4-7` | slow | high | gnarly edge cases only |

In [ ]:
# from crimellm import AnthropicClassifier
# clf = AnthropicClassifier(model="claude-haiku-4-5-20251001")
# for t in texts:
#     r = clf.classify(t)
#     print(f"{r.label:>8}  ({r.confidence:.2f})  {r.reasoning}\n          {t}")

---
## Evaluate any backend on your labeled CSV

Even with no training, **measure macro-F1**. Tells you whether the prompt is good enough to ship, or whether you need a bigger model / better prompt / labeled training data.

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score, classification_report

df = pd.read_csv(PROJECT_ROOT / "data" / "sample.csv")
id2label = {0: "no", 1: "yes", 2: "unclear"}
label2id = {v: k for k, v in id2label.items()}

results = clf.classify_many(df["text"].tolist())
y_true = df["label"].tolist()
y_pred = [label2id.get(r.label, 2) for r in results]

macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
acc = accuracy_score(y_true, y_pred)
print(f"macro_f1 = {macro:.4f}   accuracy = {acc:.4f}\n")
print(classification_report(y_true, y_pred, target_names=list(id2label.values()), zero_division=0))

In [ ]:
preds = pd.DataFrame({
    "text": df["text"],
    "truth": [id2label[i] for i in y_true],
    "pred": [r.label for r in results],
    "conf": [r.confidence for r in results],
    "reasoning": [r.reasoning for r in results],
})
preds["correct"] = preds["truth"] == preds["pred"]
preds

---
## Tuning without retraining

Bad predictions? Edit the **prompt**, not the model. Open `src/crimellm/zero_shot.py` → `SYSTEM_PROMPT`. Examples:

- Too many false `yes` → *"Default to 'no' unless the text describes a clearly illegal act."*
- Too many `unclear` → *"Avoid 'unclear' unless the text is genuinely ambiguous about who did what."*
- Domain rules → *"Treat tax evasion, money laundering, bribery as 'yes' even when described neutrally."*

Re-run the eval cell → watch macro-F1. Free A/B test.

## When to leave zero-shot

- Latency <100 ms/call → encoder probe wins (LLMs take seconds, AirLLM minutes)
- Cost <$0.0001/call → encoder probe wins
- ≥500 labels and you need reproducible decisions → fine-tune wins
- Rule too subtle to write in prose → labels capture nuance prompts can't

## Hybrid pattern (often best in production)

1. Zero-shot LLM → weak labels on your raw text
2. Human-correct a few hundred wrong ones
3. Train encoder probe on corrected labels → fast + cheap + accurate
4. Route only low-confidence items to the LLM